# Custom LLM Training for Self-Learning Chatbot

## Description
Complete training pipeline for fine-tuning Llama 3.1 8B using QLoRA for SmartSelf AI project.

## Project Overview
This notebook provides an end-to-end pipeline for:
- Loading and preprocessing training data
- Configuring QLoRA (Quantized Low-Rank Adaptation) for efficient fine-tuning
- Training a custom LLM on conversational and knowledge data
- Evaluating and visualizing training metrics
- Saving and exporting the model for deployment

## Objectives
- Fine-tune Llama 3.1 8B on custom SmartSelf AI data
- Use memory-efficient QLoRA training (4-bit quantization)
- Create a model optimized for self-learning chatbot with RAG capabilities
- Ensure reproducibility with proper seeding and logging

## CELL 2: Install Dependencies

In [ ]:
# Install required packages
!pip install transformers>=4.35.0
!pip install peft>=0.6.0
!pip install bitsandbytes>=0.41.0
!pip install accelerate>=0.24.0
!pip install datasets>=2.14.0
!pip install wandb>=0.15.0
!pip install torch>=2.0.0
!pip install sentence-transformers>=2.2.0
!pip install chromadb>=0.4.18
!pip install matplotlib>=3.7.0
!pip install scikit-learn>=1.3.0

## CELL 3: Import Libraries

In [ ]:
import torch
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, Dataset
import json
import os
from pathlib import Path
import wandb
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import warnings
import random
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility across libraries
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Make CUDA behavior deterministic where possible
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Libraries imported successfully!")
print(f"Global seed set to: {SEED}")

## CELL 4: Configuration

In [ ]:
# Model Configuration
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"
OUTPUT_DIR = "./custom_llm_model"
MAX_LENGTH = 2048

# LoRA Configuration
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training Configuration
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
WARMUP_STEPS = 100
MAX_STEPS = -1  # Set to -1 for full epochs

# Hardware Configuration
USE_4BIT = True
USE_FP16 = False
USE_BF16 = True
DEVICE_MAP = "auto"

# Data Configuration
DATA_PATH = "./data/training_data.json"
VAL_SPLIT = 0.1

print("Configuration set successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Output Directory: {OUTPUT_DIR}")

## ADDITIONAL CELL: GPU Check

Check GPU availability and memory before starting training.

In [ ]:
# Check GPU availability
print("="*50)
print("GPU AVAILABILITY CHECK")
print("="*50)

if torch.cuda.is_available():
    print(f"✓ CUDA is available")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Number of GPUs: {torch.cuda.device_count()}")
    print(f"  Current GPU: {torch.cuda.current_device()}")
    
    # Check GPU memory
    total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    allocated_memory = torch.cuda.memory_allocated(0) / (1024**3)
    cached_memory = torch.cuda.memory_reserved(0) / (1024**3)
    
    print(f"  Total GPU Memory: {total_memory:.2f} GB")
    print(f"  Allocated Memory: {allocated_memory:.2f} GB")
    print(f"  Cached Memory: {cached_memory:.2f} GB")
    print(f"  Free Memory: {total_memory - cached_memory:.2f} GB")
    
    # Check if BF16 is supported
    if torch.cuda.is_bf16_supported():
        print(f"  ✓ BF16 is supported")
    else:
        print(f"  ✗ BF16 is not supported, will use FP16")
else:
    print("✗ CUDA is not available. Training will run on CPU (slow!)")
    print("  Consider using a GPU-enabled environment for faster training.")

print("="*50)

## ADDITIONAL CELL: Data Preparation

Create sample training data if the actual data file doesn't exist.

In [ ]:
def create_sample_data(data_path):
    """Create sample training data if file doesn't exist"""
    if not os.path.exists(data_path):
        print(f"Data file not found at {data_path}")
        print("Creating sample training data...")
        
        sample_data = [
            {
                "instruction": "What is machine learning?",
                "context": "Machine learning is a subset of artificial intelligence.",
                "response": "Machine learning is a branch of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. It focuses on developing algorithms that can access data and use it to learn for themselves."
            },
            {
                "instruction": "Explain the concept of deep learning.",
                "context": "Deep learning is a subset of machine learning.",
                "response": "Deep learning is a subset of machine learning that uses neural networks with multiple layers to model complex patterns in data. It's particularly effective for tasks like image recognition, natural language processing, and speech recognition."
            },
            {
                "instruction": "What is a neural network?",
                "context": "Neural networks are computing systems inspired by biological neural networks.",
                "response": "A neural network is a computational model inspired by the structure and function of biological neural networks in the human brain. It consists of interconnected nodes or neurons that process information using connectionist approaches to computation."
            },
            {
                "instruction": "How does reinforcement learning work?",
                "context": "Reinforcement learning is a type of machine learning.",
                "response": "Reinforcement learning is a type of machine learning where an agent learns to make decisions by performing actions in an environment and receiving feedback in the form of rewards or penalties. The goal is to maximize cumulative reward over time."
            },
            {
                "instruction": "What is natural language processing?",
                "context": "NLP is a field of AI that focuses on the interaction between computers and human language.",
                "response": "Natural Language Processing (NLP) is a branch of artificial intelligence that helps computers understand, interpret, and manipulate human language. It enables machines to read text, hear speech, interpret it, measure sentiment, and determine which parts are important."
            }
        ]
        
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(data_path), exist_ok=True)
        
        # Save sample data
        with open(data_path, 'w', encoding='utf-8') as f:
            json.dump(sample_data, f, indent=2, ensure_ascii=False)
        
        print(f"Sample data created at {data_path}")
        print(f"Created {len(sample_data)} training examples")
        return sample_data
    else:
        print(f"Data file exists at {data_path}")
        return None

# Create sample data if needed
create_sample_data(DATA_PATH)

## CELL 5: Initialize Weights & Biases (Optional)

In [ ]:
# Initialize wandb for experiment tracking
# Set WANDB_DISABLED to True to disable wandb
WANDB_DISABLED = False  # Change to True to disable wandb

if not WANDB_DISABLED:
    try:
        wandb.init(
            project="smartself-llm",
            name="llama-3.1-8b-finetune",
            config={
                "model": MODEL_NAME,
                "lora_r": LORA_R,
                "learning_rate": LEARNING_RATE,
                "batch_size": BATCH_SIZE,
                "epochs": NUM_EPOCHS,
                "max_length": MAX_LENGTH,
            }
        )
        print("✓ Weights & Biases initialized successfully")
    except Exception as e:
        print(f"Warning: Could not initialize wandb: {e}")
        print("Continuing without wandb logging...")
else:
    print("Weights & Biases disabled")

## CELL 6: Load and Prepare Data

In [ ]:
def load_training_data(data_path):
    """Load training data from JSON file"""
    try:
        with open(data_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"✓ Successfully loaded data from {data_path}")
        return data
    except FileNotFoundError:
        print(f"✗ Error: Data file not found at {data_path}")
        raise
    except json.JSONDecodeError:
        print(f"✗ Error: Invalid JSON format in {data_path}")
        raise

def format_instruction(example):
    """Format data into instruction-following format"""
    instruction = example.get('instruction', '')
    context = example.get('context', '')
    response = example.get('response', '')
    
    formatted_text = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""
    
    return formatted_text

# Load data
print("Loading training data...")
raw_data = load_training_data(DATA_PATH)
print(f"Loaded {len(raw_data)} training examples")

# Display sample data
print("\nSample training example:")
print(json.dumps(raw_data[0], indent=2))

# Format data
print("\nFormatting data for training...")
formatted_data = [format_instruction(example) for example in raw_data]

# Display formatted sample
print("\nSample formatted text:")
print(formatted_data[0][:200] + "...")

# Split into train and validation
print("\nSplitting data into train and validation sets...")
train_data, val_data = train_test_split(
    formatted_data, 
    test_size=VAL_SPLIT, 
    random_state=42
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

## CELL 7: Create Dataset Objects

In [ ]:
# Create Hugging Face datasets
train_dataset = Dataset.from_dict({"text": train_data})
val_dataset = Dataset.from_dict({"text": val_data})

print("✓ Dataset objects created successfully")
print("\nTraining dataset:")
print(train_dataset)
print("\nValidation dataset:")
print(val_dataset)

# Display a sample
print("\nSample from training dataset:")
print(train_dataset[0]['text'][:300] + "...")

## CELL 8: Load Tokenizer

In [ ]:
print("Loading tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

# Set pad token if not exists
if tokenizer.pad_token is None:
    print("Setting pad token to eos token...")
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"✓ Tokenizer loaded successfully")
print(f"  Vocab size: {tokenizer.vocab_size}")
print(f"  Pad token: {tokenizer.pad_token}")
print(f"  EOS token: {tokenizer.eos_token}")
print(f"  Max model length: {tokenizer.model_max_length}")

## CELL 9: Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    """Tokenize the text data"""
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )

print("Tokenizing datasets...")

# Tokenize datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing validation data"
)

print(f"✓ Tokenization completed")
print(f"  Tokenized train dataset size: {len(tokenized_train)}")
print(f"  Tokenized val dataset size: {len(tokenized_val)}")

# Display tokenized sample
print("\nSample tokenized data:")
print(f"  Input IDs shape: {len(tokenized_train[0]['input_ids'])}")
print(f"  Attention mask shape: {len(tokenized_train[0]['attention_mask'])}")

## CELL 10: Load Base Model with Quantization

In [ ]:
print("Loading base model with quantization...")
print("This may take several minutes...")

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=USE_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if USE_4BIT else None,
    device_map=DEVICE_MAP,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    trust_remote_code=True,
)

print(f"✓ Model loaded successfully")
print(f"  Model device: {model.device}")
print(f"  Model parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"  Quantization: 4-bit" if USE_4BIT else "  Quantization: None")
print(f"  Data type: BF16" if USE_BF16 else "  Data type: FP16")

## CELL 11: Prepare Model for LoRA Training

In [ ]:
print("Preparing model for LoRA training...")

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# Configure LoRA
peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

# Apply LoRA to model
model = get_peft_model(model, peft_config)

print(f"✓ LoRA configuration applied successfully")
print(f"  LoRA Rank (r): {LORA_R}")
print(f"  LoRA Alpha: {LORA_ALPHA}")
print(f"  LoRA Dropout: {LORA_DROPOUT}")
print(f"  Target modules: {TARGET_MODULES}")

# Print trainable parameters
print("\nTrainable parameters:")
model.print_trainable_parameters()

## CELL 12: Configure Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=USE_FP16,
    bf16=USE_BF16,
    optim="paged_adamw_32bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    report_to=["wandb"] if not WANDB_DISABLED else [],
    run_name="llama-3.1-8b-finetune",
    save_total_limit=3,
    seed=42,
    data_seed=42,
)

print("✓ Training arguments configured successfully")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max steps: {MAX_STEPS if MAX_STEPS > 0 else 'Full epochs'}")

## CELL 13: Initialize Trainer

In [ ]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
    pad_to_multiple_of=8,
)

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("✓ Trainer initialized successfully")
print(f"  Training samples: {len(tokenized_train)}")
print(f"  Validation samples: {len(tokenized_val)}")

## CELL 14: Start Training

In [ ]:
print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Model: {MODEL_NAME}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Learning rate: {LEARNING_RATE}")
print("="*60)

# Start training
trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED!")
print("="*60)

## ADDITIONAL CELL: Resume Training from Checkpoint

Use this cell to resume training from a saved checkpoint if training was interrupted.

In [ ]:
# Uncomment to resume training from a checkpoint
# CHECKPOINT_PATH = "./custom_llm_model/checkpoint-XXXX"  # Replace with actual checkpoint path
# 
# print(f"Resuming training from checkpoint: {CHECKPOINT_PATH}")
# trainer.train(resume_from_checkpoint=CHECKPOINT_PATH)
# print("Training resumed and completed!")

print("To resume training from a checkpoint:")
print("1. Set CHECKPOINT_PATH to the desired checkpoint directory")
print("2. Uncomment the trainer.train() line above")
print("3. Run this cell")
print("\nAvailable checkpoints can be found in the output directory.")

## CELL 15: Plot Training Metrics

In [ ]:
print("Extracting training logs...")

# Extract training logs
logs = trainer.state.log_history

# Extract loss values
train_losses = [log['loss'] for log in logs if 'loss' in log]
eval_losses = [log['eval_loss'] for log in logs if 'eval_loss' in log]
steps = [log['step'] for log in logs if 'loss' in log]

print(f"Found {len(train_losses)} training loss entries")
print(f"Found {len(eval_losses)} validation loss entries")

# Plot training curve
if train_losses:
    plt.figure(figsize=(12, 6))
    plt.plot(steps, train_losses, label='Training Loss', alpha=0.7, linewidth=2)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training Loss Curve', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Final training loss: {train_losses[-1]:.4f}")
    print(f"Best training loss: {min(train_losses):.4f}")

# Plot validation loss if available
if eval_losses:
    eval_steps = [log['step'] for log in logs if 'eval_loss' in log]
    plt.figure(figsize=(12, 6))
    plt.plot(eval_steps, eval_losses, label='Validation Loss', alpha=0.7, color='orange', linewidth=2)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Validation Loss Curve', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print(f"Final validation loss: {eval_losses[-1]:.4f}")
    print(f"Best validation loss: {min(eval_losses):.4f}")

# Plot combined if both available
if train_losses and eval_losses:
    plt.figure(figsize=(12, 6))
    plt.plot(steps, train_losses, label='Training Loss', alpha=0.7, linewidth=2)
    plt.plot(eval_steps, eval_losses, label='Validation Loss', alpha=0.7, color='orange', linewidth=2)
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    plt.legend(fontsize=11)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## CELL 16: Save Final Model

In [ ]:
print("Saving final model...")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save the final model
final_model_path = os.path.join(OUTPUT_DIR, "final_model")
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"✓ Model saved to {final_model_path}")

# Save LoRA adapters separately
lora_path = os.path.join(OUTPUT_DIR, "lora_adapters")
model.save_pretrained(lora_path)

print(f"✓ LoRA adapters saved to {lora_path}")

# List saved files
print("\nSaved files:")
for root, dirs, files in os.walk(final_model_path):
    for file in files:
        print(f"  {os.path.join(root, file)}")

## CELL 17: Test Model Inference

In [ ]:
def generate_response(prompt, model, tokenizer, max_new_tokens=512):
    """Generate response from the fine-tuned model"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

print("Testing model inference...")

# Test with a sample prompt
test_prompt = format_instruction({
    'instruction': 'What is machine learning?',
    'context': 'Machine learning is a subset of artificial intelligence.',
    'response': ''
})

print("\nTest prompt:")
print(test_prompt)
print("\nGenerating response...")

response = generate_response(test_prompt, model, tokenizer)
print("\nGenerated Response:")
print(response)

## ADDITIONAL CELL: Batch Inference Testing

Test the model with multiple prompts to evaluate performance across different queries.

In [ ]:
def batch_inference(test_prompts, model, tokenizer, max_new_tokens=256):
    """Run batch inference on multiple prompts"""
    results = []
    
    for i, prompt_data in enumerate(test_prompts):
        print(f"\n{'='*60}")
        print(f"Test {i+1}/{len(test_prompts)}")
        print(f"{'='*60}")
        
        formatted_prompt = format_instruction(prompt_data)
        print(f"\nInstruction: {prompt_data['instruction']}")
        print(f"Context: {prompt_data['context']}")
        
        response = generate_response(formatted_prompt, model, tokenizer, max_new_tokens)
        
        # Extract just the response part
        if "### Response:" in response:
            response_text = response.split("### Response:")[-1].strip()
        else:
            response_text = response
        
        print(f"\nGenerated Response:")
        print(response_text)
        
        results.append({
            'instruction': prompt_data['instruction'],
            'context': prompt_data['context'],
            'response': response_text
        })
    
    return results

# Define test prompts
test_prompts = [
    {
        'instruction': 'Explain what deep learning is.',
        'context': 'Deep learning is a subset of machine learning.',
        'response': ''
    },
    {
        'instruction': 'How does a neural network work?',
        'context': 'Neural networks are inspired by biological brains.',
        'response': ''
    },
    {
        'instruction': 'What is natural language processing?',
        'context': 'NLP deals with human-computer language interaction.',
        'response': ''
    }
]

# Run batch inference
print("Running batch inference tests...")
batch_results = batch_inference(test_prompts, model, tokenizer)

print("\n" + "="*60)
print("Batch inference completed!")
print(f"Tested {len(batch_results)} prompts successfully")
print("="*60)

## CELL 18: Export Model for Deployment

In [ ]:
# Merge LoRA weights with base model (optional)
def merge_lora_weights(peft_model):
    """Merge LoRA adapters into the base model for a standalone deployable checkpoint."""
    print("Merging LoRA weights into base model...")
    # For PEFT models, merge_and_unload is a model method
    merged_model = peft_model.merge_and_unload()
    return merged_model

# Option 1: Save with LoRA adapters (recommended for flexibility)
print("Model saved with LoRA adapters - can be loaded with PEFT")
print(f"LoRA adapters location: {lora_path}")

# Option 2: Merge weights (optional, for deployment)
# Uncomment below to merge weights
# print("\nMerging LoRA weights with base model...")
# merged_model = merge_lora_weights(model)
# merged_model.save_pretrained(f"{OUTPUT_DIR}/merged_model")
# tokenizer.save_pretrained(f"{OUTPUT_DIR}/merged_model")
# print(f"Merged model saved to {OUTPUT_DIR}/merged_model")

print("\nDeployment options:")
print("1. Load with PEFT (recommended): Uses LoRA adapters separately")
print("2. Merge weights: Creates a single model file (larger, less flexible)")

## ADDITIONAL CELL: Integration with ChromaDB for RAG

This cell demonstrates how to integrate the fine-tuned model with ChromaDB for Retrieval-Augmented Generation.

In [ ]:
from chromadb import Client
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

print("Setting up ChromaDB integration for RAG...")

# Initialize ChromaDB client
chroma_client = Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create or get collection
collection_name = "smartself_knowledge"
try:
    collection = chroma_client.get_collection(name=collection_name)
    print(f"Found existing collection: {collection_name}")
except:
    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"description": "SmartSelf AI knowledge base"}
    )
    print(f"Created new collection: {collection_name}")

# Sample knowledge base
sample_documents = [
    "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
    "Deep learning uses neural networks with multiple layers to model complex patterns.",
    "Natural language processing enables computers to understand and generate human language.",
    "Reinforcement learning involves agents learning through trial and error with rewards.",
    "Neural networks are computing systems inspired by biological neural networks in the brain."
]

# Add or update documents in collection (safe on repeated runs)
print("\nAdding documents to knowledge base...")
doc_ids = [f"doc_{i}" for i in range(len(sample_documents))]
try:
    collection.upsert(
        documents=sample_documents,
        metadatas=[{"source": f"doc_{i}"} for i in range(len(sample_documents))],
        ids=doc_ids,
    )
    print(f"Upserted {len(sample_documents)} documents to collection")
except AttributeError:
    # Fallback for older Chroma versions that don't support upsert
    try:
        collection.add(
            documents=sample_documents,
            metadatas=[{"source": f"doc_{i}"} for i in range(len(sample_documents))],
            ids=doc_ids,
        )
        print(f"Added {len(sample_documents)} documents to collection")
    except Exception as e:
        print(f"Could not add documents directly ({e}), attempting update flow...")
        for i, doc_id in enumerate(doc_ids):
            try:
                collection.update(
                    ids=[doc_id],
                    documents=[sample_documents[i]],
                    metadatas=[{"source": doc_id}],
                )
            except Exception:
                pass
        print("Update flow completed")

# Function to retrieve relevant context
def retrieve_context(query, collection, embedding_model, n_results=2):
    """Retrieve relevant context from ChromaDB"""
    query_embedding = embedding_model.encode(query).tolist()
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    contexts = results['documents'][0]
    return contexts

# Test RAG pipeline
print("\nTesting RAG pipeline...")
query = "What is deep learning?"
contexts = retrieve_context(query, collection, embedding_model)

print(f"Query: {query}")
print(f"Retrieved context(s):")
for i, ctx in enumerate(contexts):
    print(f"  {i+1}. {ctx}")

# Generate response with RAG context
rag_prompt = format_instruction({
    'instruction': query,
    'context': ' '.join(contexts),
    'response': ''
})

print("\nGenerating response with RAG context...")
rag_response = generate_response(rag_prompt, model, tokenizer, max_new_tokens=256)
print("RAG Response:")
print(rag_response)

print("\n✓ RAG integration test completed successfully")

## CELL 19: Create Model Card

In [ ]:
model_card = f"""
# Custom LLM for SmartSelf AI

## Model Description
This is a fine-tuned version of {MODEL_NAME} for the SmartSelf AI self-learning chatbot project.

## Training Details
- **Base Model**: {MODEL_NAME}
- **Training Method**: QLoRA fine-tuning (4-bit quantization)
- **Training Epochs**: {NUM_EPOCHS}
- **Batch Size**: {BATCH_SIZE} (effective: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS})
- **Learning Rate**: {LEARNING_RATE}
- **Max Sequence Length**: {MAX_LENGTH}
- **LoRA Rank**: {LORA_R}
- **LoRA Alpha**: {LORA_ALPHA}
- **LoRA Dropout**: {LORA_DROPOUT}
- **Target Modules**: {', '.join(TARGET_MODULES)}
- **Precision**: {'BF16' if USE_BF16 else 'FP16'}

## Training Data
Trained on custom dataset containing:
- Conversational data
- Web-crawled knowledge
- API response examples
- RAG context examples

## Intended Use
Designed for self-learning chatbot with continuous knowledge acquisition capabilities.

## How to Use

### Load with PEFT (Recommended)
```python
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("{MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained("{MODEL_NAME}")

# Load LoRA adapters
model = PeftModel.from_pretrained(base_model, "{OUTPUT_DIR}/lora_adapters")

# Generate
inputs = tokenizer("Your prompt here", return_tensors="pt")
outputs = model.generate(**inputs)
```

### Load from Trainer Save
```python
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("{OUTPUT_DIR}/final_model")
tokenizer = AutoTokenizer.from_pretrained("{OUTPUT_DIR}/final_model")
```

## Limitations
- May have biases present in training data
- Should be used with appropriate safety filters
- Performance depends on quality of retrieved context in RAG setup
- Model size requires significant computational resources

## Training Configuration
- Optimizer:paged_adamw_32bit
- Learning Rate Scheduler: cosine
- Warmup Steps: {WARMUP_STEPS}
- Weight Decay: 0.01
- Gradient Accumulation: {GRADIENT_ACCUMULATION_STEPS}

## Hardware Requirements
- GPU with at least 16GB VRAM recommended for 4-bit quantization
- More VRAM required for full precision training

## License
This model inherits the license from the base model ({MODEL_NAME}).

## Citation
If you use this model, please cite:

```bibtex
@misc{{smartself_llm_2024,
  title={{Custom LLM for SmartSelf AI}},
  author={{SmartSelf AI Team}},
  year={{2024}},
  howpublished={{\url{{https://github.com/yourrepo/smartself}}}}
}}
```
"""

# Save model card
readme_path = os.path.join(OUTPUT_DIR, "README.md")
with open(readme_path, 'w', encoding='utf-8') as f:
    f.write(model_card)

print(f"✓ Model card saved to {readme_path}")

## CELL 20: Cleanup and Summary

In [ ]:
# Clear GPU cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✓ GPU cache cleared")

# Print summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Base Model: {MODEL_NAME}")
print(f"Output Directory: {OUTPUT_DIR}")
print(f"Training Epochs: {NUM_EPOCHS}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"LoRA Rank: {LORA_R}")
print(f"Max Sequence Length: {MAX_LENGTH}")

if train_losses:
    print(f"\nFinal Train Loss: {train_losses[-1]:.4f}")
    print(f"Best Train Loss: {min(train_losses):.4f}")
    
if eval_losses:
    print(f"Final Eval Loss: {eval_losses[-1]:.4f}")
    print(f"Best Eval Loss: {min(eval_losses):.4f}")

print(f"\nModel saved to: {final_model_path}")
print(f"LoRA adapters saved to: {lora_path}")
print(f"Model card saved to: {readme_path}")

print("="*60)

# Finish wandb run
if not WANDB_DISABLED:
    try:
        wandb.finish()
        print("✓ Weights & Biases run finished")
    except:
        pass

print("\nTraining pipeline completed successfully!")